# Step 1: Data Preparation for Ad Generation

**Approach: product-first, fully automated selection**

Products and topics are selected by a deterministic scoring rule — no manual curation.

**Selection rules:**
- *Eligibility:* 2+ topics with 5+ positive AND 5+ negative reviews; non-empty metadata description; exclude non-guitar accessories
- *Product score:* `n_qualifying_topics + description_length / 1000` (richness first, description quality as tiebreaker)
- *Deduplication:* one product per brand (highest score retained)
- *Final selection:* top 3 products
- *Topic selection:* per product, pick 3 qualifying topics with smallest balance gap (|pct_positive − 50%|)

**Output:** `ad_generation_input.json`

## 1. Load Review Data

In [2]:
import pandas as pd
import json

df = pd.read_parquet('../keyboards_with_topics.parquet')
print(f'Total reviews: {len(df)}')
df.head(3)

Total reviews: 92638


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,subcategory,store,average_rating,price,topic_id,topic_label,topic_prob
1,3,nice sound. pedal failed after less than 1 year,I like the piano.. but the sustain pedal faile...,[],B00723436A,B06XP6TDVY,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2019-05-22 23:22:45.290,2,True,Keyboards & MIDI,Korg,3.4,None,17.0,Durability / stopped working,0.320083
27,5,great for kids,great for kids. if you dont want to go super c...,[],B008R7F32I,B01N64DTDI,AEFLH72MFY4GRTNI3KZXNE4IUB2Q,2016-03-16 13:49:19.000,0,True,Keyboards & MIDI,Casio,4.5,None,22.0,Key feel / weighted keys,0.457159
42,4,User Friendly Keyboard,"I purchased a cheap, no name brand keyboard fo...",[],B06XWS1LLD,B074JHPL8V,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2017-08-01 22:16:11.777,4,False,Keyboards & MIDI,Casio,4.6,None,14.0,Brand quality / acoustic feel,0.348298


## 2. Load Product Metadata

Provides `product_title`, `description`, `features` for LLM prompts.  
Join key: `parent_asin`.

In [3]:
print('Loading metadata... (may take ~30 seconds)')

meta_lookup = {}
with open('../meta_Musical_Instruments.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        pa = obj.get('parent_asin')
        if pa:
            desc = obj.get('description', '')
            if isinstance(desc, list):
                desc = ' '.join(desc)
            meta_lookup[pa] = {
                'product_title': obj.get('title', ''),
                'description':   desc,
                'features':      obj.get('features', []),
                'store':         obj.get('store', ''),
                'price':         obj.get('price'),
            }

print(f'Metadata entries loaded: {len(meta_lookup)}')

Loading metadata... (may take ~30 seconds)
Metadata entries loaded: 213593


## 3. Derive Sentiment from Rating

- Rating 4–5 → **positive** (Strategy A input)
- Rating 1–2 → **negative** (Strategy B input)
- Rating 3 → dropped

In [4]:
df = df[df['rating'] != 3].copy()
df['sentiment'] = df['rating'].apply(lambda r: 'positive' if r >= 4 else 'negative')

print(f'Reviews after dropping rating=3: {len(df)}')
print(df['sentiment'].value_counts())

Reviews after dropping rating=3: 86974
sentiment
positive    75777
negative    11197
Name: count, dtype: int64


## 4. Find All Qualifying Product × Topic Combinations

A combination qualifies if it has **5+ positive AND 5+ negative** reviews.

In [5]:
MIN_PER_SENTIMENT = 5

combos = []
for (asin, topic), group in df.groupby(['asin', 'topic_label']):
    n_pos = (group['sentiment'] == 'positive').sum()
    n_neg = (group['sentiment'] == 'negative').sum()
    if n_pos >= MIN_PER_SENTIMENT and n_neg >= MIN_PER_SENTIMENT:
        combos.append({
            'asin':        asin,
            'topic':       topic,
            'n_pos':       int(n_pos),
            'n_neg':       int(n_neg),
            'total':       int(n_pos + n_neg),
            'pct_pos':     round(n_pos / (n_pos + n_neg) * 100, 1),
            'balance_gap': round(abs(n_pos / (n_pos + n_neg) - 0.5) * 100, 1),
            'parent_asin': group['parent_asin'].iloc[0],
            'store':       group['store'].iloc[0],
            'avg_rating':  group['average_rating'].iloc[0],
        })

combo_df = pd.DataFrame(combos)
print(f'Qualifying product x topic combinations: {len(combo_df)}')
print(f'Unique products: {combo_df["asin"].nunique()}')
print(f'\nTopics represented:')
print(combo_df['topic'].value_counts())

Qualifying product x topic combinations: 152
Unique products: 73

Topics represented:
topic
Returns / broken keys                 43
Power supply & accessories            34
Beginner learning                     17
Sound quality / speaker               15
Key feel / weighted keys              12
Brand quality / acoustic feel         11
Durability / stopped working          10
Connectivity (USB/MIDI/phone)          5
Appearance / color / display           2
MIDI controller / live performance     1
Synthesizer / sound design             1
Kids music lessons                     1
Name: count, dtype: int64


## 5. Automated Product Selection

Scoring rule → deduplication → top 3.

In [6]:
# --- Config ---
N_PRODUCTS          = 3    # number of products to select
N_TOPICS_PER_PROD   = 3    # number of topics per product
SAMPLE_SIZE         = 10   # max reviews to sample per topic per sentiment
EXCLUDE_ASINS       = ['B004GEW3H4']  # non-guitar accessories

# Build product-level summary
product_summary = combo_df.groupby('asin').agg(
    n_topics    = ('topic',  'count'),
    store       = ('store',  'first'),
    avg_rating  = ('avg_rating', 'first'),
    parent_asin = ('parent_asin', 'first'),
).reset_index()

product_summary['desc']     = product_summary['parent_asin'].apply(
    lambda pa: meta_lookup.get(pa, {}).get('description', ''))
product_summary['has_desc'] = product_summary['desc'].apply(
    lambda d: bool(d.strip()))
product_summary['desc_len'] = product_summary['desc'].apply(len)
product_summary['title']    = product_summary['parent_asin'].apply(
    lambda pa: meta_lookup.get(pa, {}).get('product_title', ''))

# Apply eligibility filter
eligible = product_summary[
    (product_summary['n_topics'] >= 2) &
    (product_summary['has_desc']) &
    (~product_summary['asin'].isin(EXCLUDE_ASINS))
].copy()

# Score and sort
eligible['score'] = eligible['n_topics'] + eligible['desc_len'] / 1000
eligible = eligible.sort_values('score', ascending=False).reset_index(drop=True)

# One-per-brand deduplication
selected_rows = []
used_brands   = set()
for _, row in eligible.iterrows():
    if row['store'] not in used_brands:
        selected_rows.append(row)
        used_brands.add(row['store'])
    if len(selected_rows) == N_PRODUCTS:
        break

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)

print(f'=== Selected {N_PRODUCTS} products ===')
print(selected_df[['asin', 'store', 'avg_rating', 'n_topics', 'score',
                    'title']].to_string())

=== Selected 3 products ===
         asin    store  avg_rating  n_topics  score                                                                                                                                                                 title
0  B07987K4F5   Alesis         4.6         8  9.732                 Alesis Melody 32 – Electric Keyboard Digital Piano with 32 Keys, Speakers, 300 Sounds, 300 Rhythms, 40 Songs, USB-MIDI Connectivity and Piano Lessons
1  B01AJJIQQQ  RockJam         4.5         9  9.150  RockJam 61 Key Touch Display Keyboard Piano Kit with Digital Piano Bench, Electric Piano Stand, Headphones Piano Note Stickers, Sustain Pedal & Simply Piano Lessons
2  B002KG9LWU    Casio         4.3         2  8.737                                                                                 Casio CTK-2100 61-Key Portable Keyboard Package with Headphones, Stand & Power Supply


## 6. Automated Topic Selection

For each selected product, pick the `N_TOPICS_PER_PROD` qualifying topics
with the smallest balance gap (pct_positive closest to 50%).

In [7]:
selected_topics = {}  # asin -> list of topic names

print('=== Selected topics per product ===\n')
for _, row in selected_df.iterrows():
    product_combos = combo_df[combo_df['asin'] == row['asin']].sort_values('balance_gap')
    chosen = product_combos.head(N_TOPICS_PER_PROD)
    selected_topics[row['asin']] = chosen['topic'].tolist()

    print(f"{row['asin']} | {row['store']} | {row['title'][:55]}")
    for _, c in chosen.iterrows():
        print(f"  {c['topic']:<35} pct_pos={c['pct_pos']:5.1f}%  "
              f"pos={c['n_pos']:3d}  neg={c['n_neg']:3d}  gap={c['balance_gap']:.1f}pp")
    print()

# Cross-product topic overlap
topic_sets = [set(v) for v in selected_topics.values()]
shared_all = topic_sets[0] & topic_sets[1] & topic_sets[2]
print(f'Topics shared across all 3 products: {shared_all if shared_all else "none"}')

total_ads = sum(len(v) for v in selected_topics.values()) * 2
print(f'\nTotal ads to generate: {total_ads}')

=== Selected topics per product ===

B07987K4F5 | Alesis | Alesis Melody 32 – Electric Keyboard Digital Piano with
  Sound quality / speaker             pct_pos= 61.2%  pos= 41  neg= 26  gap=11.2pp
  Brand quality / acoustic feel       pct_pos= 62.5%  pos= 15  neg=  9  gap=12.5pp
  Durability / stopped working        pct_pos= 35.0%  pos=  7  neg= 13  gap=15.0pp

B01AJJIQQQ | RockJam | RockJam 61 Key Touch Display Keyboard Piano Kit with Di
  Power supply & accessories          pct_pos= 46.6%  pos= 41  neg= 47  gap=3.4pp
  Appearance / color / display        pct_pos= 54.5%  pos=  6  neg=  5  gap=4.5pp
  Brand quality / acoustic feel       pct_pos= 60.6%  pos= 20  neg= 13  gap=10.6pp

B002KG9LWU | Casio | Casio CTK-2100 61-Key Portable Keyboard Package with He
  Returns / broken keys               pct_pos= 41.7%  pos=  5  neg=  7  gap=8.3pp
  Power supply & accessories          pct_pos= 66.7%  pos= 16  neg=  8  gap=16.7pp

Topics shared across all 3 products: none

Total ads to generate:

## 7. Sample Reviews and Build Output JSON

For each selected product × topic:
- Up to 10 positive reviews → Strategy A input
- Up to 10 negative reviews → Strategy B input

Sentiment ratios are **product-specific** (not market-wide).

In [8]:
output = []

for _, row in selected_df.iterrows():
    asin      = row['asin']
    pa        = row['parent_asin']
    meta      = meta_lookup.get(pa, {})
    group     = df[df['asin'] == asin]
    topics    = selected_topics[asin]

    product_entry = {
        'asin':           asin,
        'product_title':  meta.get('product_title', ''),
        'brand':          row['store'],
        'average_rating': float(row['avg_rating']) if pd.notna(row['avg_rating']) else None,
        'description':    meta.get('description', ''),
        'features':       meta.get('features', []),
        'topics':         {}
    }

    for topic in topics:
        topic_reviews = group[group['topic_label'] == topic]
        pos_reviews   = topic_reviews[topic_reviews['sentiment'] == 'positive']['text'].dropna()
        neg_reviews   = topic_reviews[topic_reviews['sentiment'] == 'negative']['text'].dropna()

        n_pos = len(pos_reviews)
        n_neg = len(neg_reviews)
        pct_pos = round(n_pos / (n_pos + n_neg) * 100, 1)

        product_entry['topics'][topic] = {
            'pct_positive':     pct_pos,
            'pct_negative':     round(100 - pct_pos, 1),
            'n_positive_total': int(n_pos),
            'n_negative_total': int(n_neg),
            'positive_reviews': pos_reviews.sample(
                min(SAMPLE_SIZE, n_pos), random_state=42).tolist(),
            'negative_reviews': neg_reviews.sample(
                min(SAMPLE_SIZE, n_neg), random_state=42).tolist(),
        }

    output.append(product_entry)
    print(f"Processed: {asin} | {product_entry['product_title'][:60]}")

print(f'\nTotal products: {len(output)}')
print(f'Total ads to generate: {sum(len(p["topics"])*2 for p in output)}')

Processed: B07987K4F5 | Alesis Melody 32 – Electric Keyboard Digital Piano with 32 K
Processed: B01AJJIQQQ | RockJam 61 Key Touch Display Keyboard Piano Kit with Digital
Processed: B002KG9LWU | Casio CTK-2100 61-Key Portable Keyboard Package with Headpho

Total products: 3
Total ads to generate: 16


## 8. Save Output

In [9]:
with open('ad_generation_input.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print('Saved to ad_generation_input.json\n')
print('--- Sanity Check ---')
for p in output:
    print(f"\n{p['asin']} | {p['product_title'][:55]} | ★{p['average_rating']}")
    print(f"  Desc: {p['description'][:120]}...")
    for topic, data in p['topics'].items():
        print(f"  [{topic}] "
              f"{data['n_positive_total']} pos / {data['n_negative_total']} neg total  "
              f"→ sampled {len(data['positive_reviews'])} / {len(data['negative_reviews'])}  "
              f"| pct_pos={data['pct_positive']}%")

Saved to ad_generation_input.json

--- Sanity Check ---

B07987K4F5 | Alesis Melody 32 – Electric Keyboard Digital Piano with | ★4.6
  Desc: " The Ultimate Go-Anywhere Portable Electric Keyboard – Introducing the Melody 32 from Alesis Simplify learning the digi...
  [Sound quality / speaker] 41 pos / 26 neg total  → sampled 10 / 10  | pct_pos=61.2%
  [Brand quality / acoustic feel] 15 pos / 9 neg total  → sampled 10 / 9  | pct_pos=62.5%
  [Durability / stopped working] 7 pos / 13 neg total  → sampled 7 / 10  | pct_pos=35.0%

B01AJJIQQQ | RockJam 61 Key Touch Display Keyboard Piano Kit with Di | ★4.5
  Desc: The ROCKJAM 761 piano keyboard Super Kit is a complete package for any aspiring pianist containing all you need to take ...
  [Power supply & accessories] 41 pos / 47 neg total  → sampled 10 / 10  | pct_pos=46.6%
  [Appearance / color / display] 6 pos / 5 neg total  → sampled 6 / 5  | pct_pos=54.5%
  [Brand quality / acoustic feel] 20 pos / 13 neg total  → sampled 10 / 10  | pct_pos

## 9. Preview One Entry

Verify the JSON structure before moving to Step 2 (ad generation).

In [10]:
first       = output[0]
first_topic = list(first['topics'].keys())[0]
data        = first['topics'][first_topic]

print(f"Product : {first['product_title']}")
print(f"Brand   : {first['brand']}  |  Rating: {first['average_rating']}")
print(f"\nDescription:\n{first['description'][:400]}")
print(f"\nFeatures:")
for feat in first['features'][:3]:
    print(f'  • {feat}')

print(f"\n{'='*60}")
print(f"Topic: {first_topic}")
print(f"Product-level sentiment: {data['pct_positive']}% pos / {data['pct_negative']}% neg")
print(f"Total in topic: {data['n_positive_total']} pos, {data['n_negative_total']} neg")

print(f"\n--- Positive reviews (Strategy A input) ---")
for r in data['positive_reviews'][:2]:
    print(f'  • {r[:250]}')

print(f"\n--- Negative reviews (Strategy B input) ---")
for r in data['negative_reviews'][:2]:
    print(f'  • {r[:250]}')

Product : Alesis Melody 32 – Electric Keyboard Digital Piano with 32 Keys, Speakers, 300 Sounds, 300 Rhythms, 40 Songs, USB-MIDI Connectivity and Piano Lessons
Brand   : Alesis  |  Rating: 4.6

Description:
" The Ultimate Go-Anywhere Portable Electric Keyboard – Introducing the Melody 32 from Alesis Simplify learning the digital piano with the Melody 32 from Alesis. This ultra-portable piano packs an arsenal of premium features into a back-pack friendly keyboard, that’s ready to make music wherever you go, including: 300 high-quality digital keyboard sounds (including traditional piano, orchestral in

Features:
  • Feature Packed Digital Piano for Kids – Portable electronic keyboard with 32 premium mini piano style keys with power via USB or 4 AA batteries (not included)
  • Premium Electric Piano Keyboard Sounds - 300 voices (including Acoustic Piano, Electric Piano, Strings, Organ, Synth, Drums and much more); built in speakers that deliver room-filling sound
  • Practice Makes Perfe